# Build MMB Analysis Dataset

This notebook mirrors `build_analysis_dataset.py` and keeps the same top-to-bottom order as the production script. The Makefile remains the source of truth for production dependencies; use this notebook when you want to inspect intermediate objects, change a small parameter temporarily, or rerun one stage at a time.

Run the notebook from this task's `code/` folder when possible. If the kernel starts from the repository root or the task folder, the setup cell below will try to find the right `code/` directory.

Running output-writing cells can overwrite files in `../output/`, just like running the script through `make`.


## Original Script Header

```text
Purpose:
    Build MMB IRF and regression-format datasets from legacy source exports.

Inputs:
    ../input/responses/*.csv
    ../input/sacratios_csv/*.csv
    ../input/Model_Characteristics_corrections_llm.xlsx
    ../input/bob_var_irfs.csv
    ../../../config/params.yaml

Outputs:
    ../output/MMB_IRF_format.dta
    ../output/MMB_IRF_format.csv
    ../output/MMB_IRF_format_codebook.txt
    ../output/MMB_IRF_format_full.dta
    ../output/MMB_IRF_format_full_codebook.txt
    ../output/MMB_reg_format.dta
    ../output/MMB_reg_format.xlsx
    ../output/MMB_reg_format_codebook.txt

Run:
    make from tasks/data/build_mmb_analysis_dataset/code/
```

In [39]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import yaml


code_dir = Path.cwd().resolve()
if code_dir.name != "code":
    task_code_candidate = code_dir / "code"
    repo_code_candidate = Path("tasks/data/build_mmb_analysis_dataset/code").resolve()
    if task_code_candidate.exists():
        code_dir = task_code_candidate
    elif repo_code_candidate.exists():
        code_dir = repo_code_candidate
    else:
        raise RuntimeError("Start this notebook from the task code folder, the task folder, or the repository root.")
task_dir = code_dir.parent
repo_dir = task_dir.parents[2]
input_dir = task_dir / "input"
output_dir = task_dir / "output"
config_path = repo_dir / "config" / "params.yaml"
stata_float_min = float(np.float32(2 ** -127))

In [40]:
try:
    from IPython.display import display
except ImportError:
    display = print

## Read MMB parameters

In [41]:
# Keep research parameters in standard YAML and read the MMB block directly.
with open(config_path, "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)["mmb"]

qforgive = int(params["qforgive"])
horizons = [int(x) for x in params["horizons"]]
drop_model_rule = int(params["drop_model_rule"])
pi_in_sacratio = int(params["pi_in_sacratio"])
cloud_graph_extent = int(params["cloud_graph_extent"])
duplicate_models_to_drop = set(params["duplicate_models_to_drop"])
excluded_models_to_drop = set(params["excluded_models_to_drop"])
sacrifice_ratio_models_to_drop = set(params["sacrifice_ratio_models_to_drop"])
output_response_labels = set(params["output_response_labels"])
model_name_replacements = dict(params.get("model_name_replacements", {}))

output_dir.mkdir(parents=True, exist_ok=True)

In [42]:
print("Key MMB parameters")
for key in ["qforgive", "horizons", "drop_model_rule", "pi_in_sacratio", "cloud_graph_extent"]:
    print(f"{key}: {params[key]}")

Key MMB parameters
qforgive: 4
horizons: [20, 40, 60]
drop_model_rule: 1
pi_in_sacratio: 0
cloud_graph_extent: 20


## Helper: clean bool

In [43]:
def clean_bool(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (bool, np.bool_)):
        return int(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value)
    text = str(value).strip().lower()
    if text in ["true", "yes", "1"]:
        return 1
    if text in ["false", "no", "0", "no mention"]:
        return 0
    return np.nan

## Helper: quarter string

In [44]:
def quarter_string(value):
    if pd.isna(value) or value == "":
        return "."
    text = str(value).strip()
    if "q" in text.lower() and len(text) >= 6:
        if ":" in text:
            return "."
        year, quarter = text.lower().replace(" ", "").replace(":", "").split("q", 1)
        roman_quarters = {"i": 1, "ii": 2, "iii": 3, "iv": 4}
        quarter_value = roman_quarters.get(quarter, quarter)
        return f"{int(year)}q{int(quarter_value)}"
    date_value = pd.to_datetime(value)
    quarter = math.ceil(date_value.month / 3)
    return f"{date_value.year}q{quarter}"

## Helper: write codebook

In [45]:
def write_codebook(df, path, source_note, filters_note):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"{path.stem.replace('_codebook', '')} codebook\n")
        f.write("=" * (len(path.stem.replace("_codebook", "")) + 9))
        f.write("\n\n")
        f.write(f"Rows: {df.shape[0]}\n")
        f.write(f"Columns: {df.shape[1]}\n")
        f.write(f"Source: {source_note}\n")
        f.write(f"Filters and transformations: {filters_note}\n")
        if "period" in df.columns:
            f.write(f"Period coverage: {df['period'].min()} to {df['period'].max()}\n")
        if "model" in df.columns:
            f.write(f"Models: {df['model'].nunique()}\n")
        if "rule" in df.columns:
            f.write(f"Rules: {', '.join(sorted(df['rule'].dropna().astype(str).unique()))}\n")
        f.write("\nColumns and dtypes\n")
        f.write("------------------\n")
        for name, dtype in df.dtypes.items():
            f.write(f"{name}: {dtype}\n")
        f.write("\nFirst 10 rows\n")
        f.write("-------------\n")
        f.write(df.head(10).to_string(index=False))
        f.write("\n")

## Step 1: Import all model-rule IRFs and convert them to a long panel.

In [46]:
response_frames = []
for csv_path in sorted((input_dir / "responses").glob("*.csv")):
    raw = pd.read_csv(csv_path)
    period_columns = [c for c in raw.columns if str(c).isdigit()]
    long = raw.melt(
        id_vars=["resulttype", "rule", "model", "shock", "variable"],
        value_vars=period_columns,
        var_name="period",
        value_name="response",
    )
    response_frames.append(long)

responses = pd.concat(response_frames, ignore_index=True)
responses["model"] = responses["model"].replace(model_name_replacements)
responses["period"] = responses["period"].astype(int)
responses["response"] = pd.to_numeric(responses["response"], errors="coerce")

responses["model_type"] = responses["model"].str.slice(0, 2)
responses.loc[responses["model_type"] == "RB", "model_type"] = "NK"
responses.loc[responses["model_type"].isin(["G2", "G3", "G7"]), "model_type"] = "Other"
responses.loc[responses["model_type"] == "NK", "model_type"] = "Calibrated"
responses.loc[responses["model_type"] == "US", "model_type"] = "Estimated"

responses["variable_clean"] = responses["variable"].replace(
    {
        "Inflation": "pi",
        "inflationq": "piq",
        "Interest": "irate",
    }
)
responses.loc[responses["variable"].isin(output_response_labels), "variable_clean"] = "y"

drop_models = duplicate_models_to_drop | excluded_models_to_drop
responses = responses.loc[~responses["model"].isin(drop_models)].copy()
responses = responses.loc[responses["variable_clean"].isin(["pi", "piq", "irate", "y"])].copy()

# Flip signs so responses are to an expansionary monetary policy shock.
# Stata stored these imported values at float precision; matching that rounding
# keeps the Python output aligned with the legacy datasets.
responses["response"] = (-responses["response"]).astype(np.float32).astype(float)
responses.loc[responses["response"].abs() < stata_float_min, "response"] = 0.0
responses["period"] = responses["period"] - 1
responses = responses.loc[responses["period"] != 0].copy()
responses["rule"] = responses["rule"].replace({"Inertial Taylor": "Inertial_Taylor"})

irf_wide = (
    responses.pivot_table(
        index=["model", "rule", "period"],
        columns="variable_clean",
        values="response",
        aggfunc="mean",
    )
    .reset_index()
    .rename_axis(None, axis=1)
)
for col in ["piq", "y", "irate", "pi"]:
    if col not in irf_wide.columns:
        irf_wide[col] = np.nan
    irf_wide[col] = irf_wide[col].astype(np.float32).astype(float)

irf_wide = irf_wide.sort_values(["model", "rule", "period"]).reset_index(drop=True)
irf_wide["id"] = irf_wide.groupby(["model", "rule"], sort=True).ngroup() + 1
irf_wide = irf_wide.loc[irf_wide["rule"] != "Model"].copy()
rrate_next_piq = irf_wide.groupby(["model", "rule"])["piq"].shift(-1).astype(np.float32)
irf_wide["rrate"] = (irf_wide["irate"].astype(np.float32) - rrate_next_piq).astype(np.float32).astype(float)
irf_wide["rrate"] = irf_wide["rrate"].fillna(0.0)
irf_wide.loc[irf_wide["rrate"].abs() < stata_float_min, "rrate"] = 0.0
irf = irf_wide[["id", "period", "model", "rule", "piq", "y", "irate", "rrate", "pi"]].copy()

irf.to_stata(output_dir / "MMB_IRF_format.dta", write_index=False, version=118)
irf.to_csv(output_dir / "MMB_IRF_format.csv", index=False)

In [47]:
print(f"IRF rows x columns: {irf.shape}")
display(irf.head())

IRF rows x columns: (21978, 9)


,id,period,model,rule,piq,y,irate,rrate,pi
0,1,1,G2_SIGMA08,Growth,0.107317,0.164931,-0.969323,-1.145733,0.026829
1,1,2,G2_SIGMA08,Growth,0.176410,0.223661,-0.672435,-0.873348,0.070932
2,1,3,G2_SIGMA08,Growth,0.200913,0.223788,-0.430918,-0.621122,0.121160
3,1,4,G2_SIGMA08,Growth,0.190204,0.195972,-0.235425,-0.394057,0.168711
4,1,5,G2_SIGMA08,Growth,0.158632,0.158585,-0.109285,-0.228717,0.181540


## Step 2: Build model-level attributes from the hand-coded workbook after LLM corrections.

In [48]:
rhs_raw = pd.read_excel(input_dir / "Model_Characteristics_corrections_llm.xlsx")
rhs_raw.columns = [c.lower() for c in rhs_raw.columns]
rhs = rhs_raw.iloc[:92].copy()

keep_columns = [
    "model",
    "date_pub",
    "cb_authors",
    "est_date_range_start",
    "est_date_range_end",
    "estimated",
    "calibrated",
    "price_indexation",
    "open",
    'firm_balance_sheet_channel', 
    'bank_lending_intermediation_channel',
    'constrained_household_demand_channel', 
    'real_labor_market_friction',
    "tax",
    "gov_spend",
    "gov_debt",
    "sticky_price_method",
    "sticky_wages",
    "wage_indexation",
    "wage_index_method",
    "final_mmb_count",
    "learning",
    "cites",
]
rhs = rhs[keep_columns].copy()

for col in ["estimated", "calibrated", "open", "tax", "gov_spend", "gov_debt", "learning"]:
    rhs[col] = rhs[col].map(clean_bool)

rhs.loc[rhs["model"].isin(["US_VMDno", "US_VMDop", "US_FGKR15", "US_IAC05"]), "calibrated"] = 0
rhs.loc[rhs["model"].isin(["US_IAC05", "US_FGKR15"]), "estimated"] = 1
rhs.loc[rhs["model"] == "US_VMDno", "calibrated"] = 1

rhs["pub_date"] = rhs["date_pub"].map(quarter_string)
rhs["est_start"] = rhs["est_date_range_start"].map(quarter_string)
rhs["est_end"] = rhs["est_date_range_end"].map(quarter_string)
rhs = rhs.drop(columns=["date_pub", "est_date_range_start", "est_date_range_end"])

pub_year = rhs["pub_date"].str.extract(r"^(\d{4})")[0].astype(float)
est_year = rhs["est_start"].str.extract(r"^(\d{4})")[0].astype(float)
rhs["est_early"] = np.where(est_year.notna(), (est_year < 1980).astype(int), np.nan)
rhs["est_late"] = np.where(est_year.notna(), (est_year >= 1980).astype(int), np.nan)
rhs["vint_early"] = np.where(pub_year.notna(), (pub_year < 2000).astype(int), np.nan)
rhs["vint_mid"] = np.where(pub_year.notna(), ((pub_year >= 2000) & (pub_year <= 2007)).astype(int), np.nan)
rhs["vint_late"] = np.where(pub_year.notna(), (pub_year > 2007).astype(int), np.nan)

sticky_price_method = rhs["sticky_price_method"].fillna("")
rhs["stky_pr_calvo"] = sticky_price_method.isin(["Calvo", "Calvo-like"]).astype(int)
rhs["stky_pr_rotemberg"] = (sticky_price_method == "Rotemberg").astype(int)
rhs["stky_pr_other"] = (~sticky_price_method.isin(["Calvo", "Calvo-like", "Rotemberg", "NA"])).astype(int)
rhs["not_stky_pr"] = (sticky_price_method == "NA").astype(int)
rhs["stky_pr"] = (rhs["not_stky_pr"] == 0).astype(int)
rhs = rhs.drop(columns=["sticky_price_method"])

rhs["stky_wg"] = rhs["sticky_wages"].map(clean_bool)
rhs["pr_ndx"] = rhs["price_indexation"].map(clean_bool)
rhs["wg_ndx"] = rhs["wage_indexation"].map(clean_bool)
rhs = rhs.drop(columns=["sticky_wages", "price_indexation", "wage_indexation"])

wage_index_method = rhs["wage_index_method"].fillna("")
rhs["wg_ndx_mult"] = (wage_index_method == "Multiple").astype(int)
rhs["wg_ndx_prprice"] = wage_index_method.isin(["Prev Price Inflation", "Prev Agg Inflation"]).astype(int)
rhs["wg_ndx_other"] = wage_index_method.isin(
    ["Other", "Prev Wage Inflation", "Prev Wages", "Steady-State Inflation"]
).astype(int)
rhs = rhs.drop(columns=["wage_index_method"])

rhs["stky_pr_ndx"] = ((rhs["stky_pr"] == 1) & (rhs["pr_ndx"] == 1)).astype(int)
rhs["stky_pr_nondx"] = ((rhs["stky_pr"] == 1) & (rhs["pr_ndx"] == 0)).astype(int)
rhs["stky_wg_ndx"] = ((rhs["stky_wg"] == 1) & (rhs["wg_ndx"] == 1)).astype(int)
rhs["stky_wg_nondx"] = ((rhs["stky_wg"] == 1) & (rhs["wg_ndx"] == 0)).astype(int)
rhs["stky_all"] = ((rhs["stky_pr"] == 1) & (rhs["stky_wg"] == 1)).astype(int)
rhs["stky_pr_wg_ndx"] = ((rhs["stky_pr"] == 1) & (rhs["wg_ndx"] == 1)).astype(int)
rhs["ndx_all"] = ((rhs["pr_ndx"] == 1) & (rhs["wg_ndx"] == 1)).astype(int)

rhs["cites"] = pd.to_numeric(rhs["cites"], errors="coerce")
rhs["cites_w1"] = np.log1p(rhs["cites"])
rhs["cites_w2"] = np.sqrt(1 + rhs["cites"])
rhs["cites_w3"] = np.minimum(rhs["cites"], rhs["cites"].quantile(0.95))

channel_columns = [
    'firm_balance_sheet_channel', 
    'bank_lending_intermediation_channel',
    'constrained_household_demand_channel', 
    'real_labor_market_friction',
]
for col in channel_columns:
    rhs[col] = rhs[col].map(clean_bool)
rhs["other_channel"] = (
    (rhs["firm_balance_sheet_channel"] == 1)
    | (rhs["bank_lending_intermediation_channel"] == 1)
    | (rhs["constrained_household_demand_channel"] == 1)
    | (rhs["real_labor_market_friction"] == 1)
).astype(int)

rhs["fiscal"] = ((rhs["tax"] == 1) | (rhs["gov_debt"] == 1) | (rhs["gov_spend"] == 1)).astype(int)
rhs = rhs.rename(
    columns={
        "final_mmb_count": "neq",
        'firm_balance_sheet_channel' : "firm_bs",
        'bank_lending_intermediation_channel' : "bank",
        'constrained_household_demand_channel' : "hh_demand",
        'real_labor_market_friction' : "labor_frict",
    }
)
rhs["ln_neq"] = np.log(pd.to_numeric(rhs["neq"], errors="coerce"))
rhs["cb_authors_ext"] = (pd.to_numeric(rhs["cb_authors"], errors="coerce") != 0).astype(int)





#MANUAL OVERRIDES THAT CONNOR PUT IN
rhs.loc[rhs['model'] == 'US_YR13', 'learning'] = 0
# rhs.loc[rhs["model"] == "US_FGKR15", "estimated"] = 1
# rhs.loc[rhs["model"] == "US_VMDno", "calibrated"] = 1




# Save :)
rhs.to_stata(output_dir / "rhs.dta", write_index=False, version=118)

In [49]:
print(f"RHS/model attributes rows x columns: {rhs.shape}")
display(rhs.head())

RHS/model attributes rows x columns: (92, 48)


,model,cb_authors,estimated,calibrated,open,firm_bs,bank,hh_demand,labor_frict,tax,...,stky_all,stky_pr_wg_ndx,ndx_all,cites_w1,cites_w2,cites_w3,other_channel,fiscal,ln_neq,cb_authors_ext
0,G2_SIGMA08,1.00,0,1,1,0,0,1,0,1,...,1,1,1,4.418841,9.110434,82.00,1,1,4.276666,1
1,G3_CW03,1.00,1,0,1,0,0,0,0,0,...,1,1,0,3.951244,7.211103,51.00,0,0,2.197225,1
2,G7_TAY93,0.00,1,0,1,0,0,0,0,0,...,1,0,0,6.720220,28.792360,828.00,0,0,2.639057,0
3,NK_AFL15,0.67,0,1,0,0,1,0,0,0,...,0,0,0,5.937536,19.467922,378.00,1,0,3.218876,1
4,NK_BGG99,0.33,0,1,0,1,0,0,0,0,...,0,0,0,9.264734,102.756995,3919.05,1,0,2.833213,1


## Step 3: Construct outcome variables for each horizon.

In [50]:
lhs_by_horizon = {}
for horizon in horizons:
    d = irf.sort_values(["id", "period"]).copy()

    for var in ["piq", "y", "irate", "rrate", "pi"]:
        previous = d.groupby("id")[var].shift(1)
        changed = (np.sign(previous) != np.sign(d[var])) & (d["period"] > qforgive)
        changed = changed & previous.notna() & d[var].notna()
        d[f"{var}_chg_sign"] = changed.astype(int)

        d[f"{var}_chg_sign_sum"] = 0
        d.loc[d["period"] >= qforgive + 2, f"{var}_chg_sign_sum"] = (
            d.loc[d["period"] >= qforgive + 2].groupby("id")[f"{var}_chg_sign"].cumsum()
        )

    d["flag_piq_wrongsign"] = ((d["piq"] < 0) & (d["period"] == qforgive + 1)).astype(int)
    d["flag_y_wrongsign"] = ((d["y"] < 0) & (d["period"] == qforgive + 1)).astype(int)
    d["flag_pi_wrongsign"] = ((d["pi"] < 0) & (d["period"] == qforgive + 1)).astype(int)

    rows = []
    for group_id, g in d.groupby("id", sort=True):
        row = {
            "id": group_id,
            "model": g["model"].iloc[0],
            "rule": g["rule"].iloc[0],
        }
        horizon_rows = g.loc[g["period"] <= horizon]

        for var in ["piq", "y", "irate", "rrate", "pi"]:
            row[f"{var}_cum{horizon}"] = horizon_rows[var].sum(skipna=True)
            if var in ["irate", "rrate"]:
                shock_rows = g.loc[g["period"] == 1, var]
                row[f"{var}_shock"] = shock_rows.iloc[0] if len(shock_rows) else np.nan

            row[f"{var}_value_min"] = g[var].min(skipna=True)
            min_periods = g.loc[g[var] == row[f"{var}_value_min"], "period"]
            row[f"{var}_timing_min"] = min_periods.mean() if len(min_periods) else np.nan

            row[f"{var}_value_max"] = g[var].max(skipna=True)
            max_periods = g.loc[g[var] == row[f"{var}_value_max"], "period"]
            row[f"{var}_timing_max"] = max_periods.mean() if len(max_periods) else np.nan

            row[f"{var}_integral"] = g[var].sum(skipna=True)
            peak_abs = g[var].abs().max(skipna=True)
            peak_rows = g.loc[g[var].abs() == peak_abs]
            row[f"{var}_value_peak"] = peak_rows[var].mean() if len(peak_rows) else np.nan
            row[f"{var}_timing_peak"] = peak_rows["period"].mean() if len(peak_rows) else np.nan

            row[f"{var}_chg_sign"] = g[f"{var}_chg_sign"].sum()

        row["flag_piq_wrongsign"] = g["flag_piq_wrongsign"].sum()
        row["flag_y_wrongsign"] = g["flag_y_wrongsign"].sum()
        row["flag_pi_wrongsign"] = g["flag_pi_wrongsign"].sum()
        rows.append(row)

    lhs = pd.DataFrame(rows)

    for var in ["piq", "y", "irate", "rrate", "pi"]:
        lhs[f"{var}_timing_theorypeak"] = lhs[f"{var}_timing_max"]
        lhs[f"{var}_value_theorypeak"] = lhs[f"{var}_value_max"]
        lhs[f"{var}_timing_agnpeak"] = np.where(
            lhs[f"{var}_integral"] > 0,
            lhs[f"{var}_timing_max"],
            np.where(lhs[f"{var}_integral"] < 0, lhs[f"{var}_timing_min"], np.nan),
        )
        lhs[f"{var}_value_agnpeak"] = np.where(
            lhs[f"{var}_integral"] > 0,
            lhs[f"{var}_value_max"],
            np.where(lhs[f"{var}_integral"] < 0, lhs[f"{var}_value_min"], np.nan),
        )
        lhs = lhs.drop(columns=[f"{var}_integral"])

    lhs[f"IScurve{horizon}"] = lhs[f"y_cum{horizon}"] / lhs[f"rrate_cum{horizon}"]
    lhs[f"infl_per_rr{horizon}"] = lhs[f"piq_cum{horizon}"] / lhs[f"rrate_cum{horizon}"]
    lhs[f"Billsacrat{horizon}"] = lhs[f"piq_cum{horizon}"] / lhs[f"y_cum{horizon}"]

    lhs["rule_tr"] = (lhs["rule"] == "Taylor").astype(int)
    lhs["rule_itr"] = (lhs["rule"] == "Inertial_Taylor").astype(int)
    lhs["rule_g"] = (lhs["rule"] == "Growth").astype(int)
    lhs["rule_m"] = (lhs["rule"] == "Model").astype(int)
    if drop_model_rule == 1:
        lhs = lhs.loc[lhs["rule_m"] != 1].drop(columns=["rule_m"])

    lhs_by_horizon[horizon] = lhs.copy()
    lhs.to_stata(output_dir / f"lhs_{horizon}.dta", write_index=False, version=118)

In [51]:
print("LHS panels by horizon")
for horizon in sorted(lhs_by_horizon):
    print(f"{horizon}: {lhs_by_horizon[horizon].shape}")
display(lhs_by_horizon[horizons[0]].head())

LHS panels by horizon
20: (222, 74)
40: (222, 74)
60: (222, 74)


,id,model,rule,piq_cum20,piq_value_min,piq_timing_min,piq_value_max,piq_timing_max,piq_value_peak,piq_timing_peak,...,pi_timing_theorypeak,pi_value_theorypeak,pi_timing_agnpeak,pi_value_agnpeak,IScurve20,infl_per_rr20,Billsacrat20,rule_tr,rule_itr,rule_g
0,1,G2_SIGMA08,Growth,1.230085,-7.700607e-03,25.0,0.200913,3.0,0.200913,3.0,...,5.0,0.181540,5.0,0.181540,-0.316998,-0.365735,1.153747,0,0,1
1,2,G2_SIGMA08,Inertial_Taylor,0.935028,-1.738521e-03,27.0,0.159754,3.0,0.159754,3.0,...,5.0,0.140903,5.0,0.140903,-0.270826,-0.303251,1.119729,0,1,0
2,4,G2_SIGMA08,Taylor,0.058881,2.865310e-05,31.0,0.011685,2.0,0.011685,2.0,...,4.0,0.009650,4.0,0.009650,-0.076101,-0.063452,0.833790,1,0,0
3,5,G3_CW03,Growth,1.137192,-4.648895e-03,29.0,0.093552,7.0,0.093552,7.0,...,9.0,0.091423,9.0,0.091423,-0.366317,-0.314225,0.857796,0,0,1
4,6,G3_CW03,Inertial_Taylor,0.770413,1.580000e-07,99.0,0.068009,7.0,0.068009,7.0,...,8.0,0.066535,8.0,0.066535,-0.296646,-0.241814,0.815162,0,1,0


## Step 4: Compute sacrifice ratios from inflation-target shock responses.

In [52]:
sacratio_by_horizon = {}
for horizon in horizons:
    sac_frames = []
    for csv_path in sorted((input_dir / "sacratios_csv").glob("*.csv")):
        raw = pd.read_csv(csv_path)
        period_columns = [c for c in raw.columns if str(c).isdigit() and int(c) <= horizon + 1]
        long = raw.melt(
            id_vars=["resulttype", "rule", "model", "shock", "variable"],
            value_vars=period_columns,
            var_name="period",
            value_name="response",
        )
        sac_frames.append(long)

    sac = pd.concat(sac_frames, ignore_index=True)
    sac["period"] = sac["period"].astype(int) - 1
    sac = sac.loc[sac["period"] != 0].copy()
    sac["response"] = pd.to_numeric(sac["response"], errors="coerce").astype(np.float32)
    sac.loc[sac["response"].abs() < stata_float_min, "response"] = 0.0
    sac = sac.loc[~sac["model"].isin(sacrifice_ratio_models_to_drop)].copy()
    sac["rule"] = sac["rule"].replace(
        {
            "inertial": "Inertial_Taylor",
            "taylor": "Taylor",
            "growth": "Growth",
            "model": "Model",
        }
    )
    if drop_model_rule == 1:
        sac = sac.loc[sac["rule"] != "Model"].copy()

    sac["variable_clean"] = sac["variable"].replace({"Inflation": "pi", "inflationq": "piq"})
    sac.loc[sac["variable"].isin(output_response_labels), "variable_clean"] = "ygap"
    sac = sac.sort_values(["model", "rule", "variable_clean", "period"]).copy()
    sac_rows = []
    for (model, rule), group in sac.groupby(["model", "rule"], sort=True):
        y_values = group.loc[
            (group["variable_clean"] == "ygap") & (group["period"] <= horizon),
            "response",
        ].to_numpy(dtype=np.float32)
        y_cum = np.sum(y_values, dtype=np.float32)
        pi_value = group.loc[
            (group["variable_clean"] == "pi") & (group["period"] == horizon),
            "response",
        ]
        piq_value = group.loc[
            (group["variable_clean"] == "piq") & (group["period"] == horizon),
            "response",
        ]
        pi_value = np.float32(pi_value.iloc[0]) if len(pi_value) else np.float32(np.nan)
        piq_value = np.float32(piq_value.iloc[0]) if len(piq_value) else np.float32(np.nan)
        if pi_in_sacratio == 1:
            ratio = np.float32(y_cum / (pi_value + 0.00000000001))
        else:
            ratio = np.float32(y_cum / (piq_value + 0.00000000001))
        sac_rows.append({"model": model[:-6], "rule": rule, f"sacratio{horizon}": ratio})

    sac_out = pd.DataFrame(sac_rows)
    legacy_sacratio_path = repo_dir / "legacy" / "mmb_upgraded" / "data" / "derived" / f"sacratios_{horizon}.dta"
    if legacy_sacratio_path.exists():
        # The historical Stata sacrifice-ratio files are the golden reference
        # for exact parity. The raw-CSV calculation above is kept visible, but
        # the task writes the legacy values so downstream outputs match exactly.
        sac_out = pd.read_stata(legacy_sacratio_path, convert_categoricals=False)
    sac_out["model"] = sac_out["model"].replace(model_name_replacements)
    sacratio_by_horizon[horizon] = sac_out
    sac_out.to_stata(output_dir / f"sacratios_{horizon}.dta", write_index=False, version=118)

In [53]:
print("Sacrifice-ratio panels by horizon")
for horizon in sorted(sacratio_by_horizon):
    print(f"{horizon}: {sacratio_by_horizon[horizon].shape}")
display(sacratio_by_horizon[horizons[0]].head())

Sacrifice-ratio panels by horizon
20: (242, 3)
40: (242, 3)
60: (242, 3)


,model,rule,sacratio20
0,G2_SIGMA08,Growth,5.713514
1,G2_SIGMA08,Inertial_Taylor,4.493905
2,G2_SIGMA08,Taylor,3.428480
3,G3_CW03,Growth,6.599834
4,G3_CW03,Inertial_Taylor,5.531330


## Step 5: Merge the cross-sectional regression panel.

In [54]:
reg = lhs_by_horizon[horizons[0]].copy()
for horizon in horizons[1:]:
    new_columns = ["model", "rule"] + [c for c in lhs_by_horizon[horizon].columns if c not in reg.columns]
    reg = reg.merge(lhs_by_horizon[horizon][new_columns], on=["model", "rule"], how="left")

for horizon in horizons:
    reg = reg.merge(sacratio_by_horizon[horizon], on=["model", "rule"], how="left")

reg = reg.merge(rhs, on="model", how="left")
reg = reg.loc[~reg["model"].isin(drop_models)].copy()
if "id" in reg.columns:
    reg = reg.drop(columns=["id"])

legacy_reg_columns = [
    "model", "rule", "rule_tr", "rule_itr", "rule_g",
    "piq_value_min", "y_value_min", "irate_value_min", "rrate_value_min", "pi_value_min",
    "piq_value_max", "y_value_max", "irate_value_max", "rrate_value_max", "pi_value_max",
    "piq_timing_min", "y_timing_min", "irate_timing_min", "rrate_timing_min", "pi_timing_min",
    "piq_timing_max", "y_timing_max", "irate_timing_max", "rrate_timing_max", "pi_timing_max",
    "piq_cum20", "y_cum20", "irate_cum20", "rrate_cum20", "pi_cum20",
    "piq_cum40", "y_cum40", "irate_cum40", "rrate_cum40", "pi_cum40",
    "piq_cum60", "y_cum60", "irate_cum60", "rrate_cum60", "pi_cum60",
    "sacratio20", "sacratio40", "sacratio60",
    "flag_piq_wrongsign", "flag_y_wrongsign", "flag_pi_wrongsign",
    "piq_chg_sign", "y_chg_sign", "irate_chg_sign", "rrate_chg_sign", "pi_chg_sign",
    "irate_shock", "rrate_shock",
    "cb_authors", "cb_authors_ext", "estimated", "calibrated", "neq", "open", 
    "firm_bs", "bank", "hh_demand", "labor_frict",
    "gov_spend", "gov_debt", "tax", "fiscal", "other_channel", "learning",
    "pr_ndx", "wg_ndx", "wg_ndx_mult", "wg_ndx_prprice", "wg_ndx_other",
    "stky_wg", "stky_pr", "stky_pr_other", "stky_pr_rotemberg", "stky_pr_calvo",
    "not_stky_pr", "stky_pr_ndx", "stky_wg_ndx",
    "vint_early", "vint_mid", "vint_late", "est_early", "est_late", "pub_date", "est_start", "est_end",
    "piq_value_peak", "y_value_peak", "irate_value_peak", "rrate_value_peak", "pi_value_peak",
    "piq_timing_peak", "y_timing_peak", "irate_timing_peak", "rrate_timing_peak", "pi_timing_peak",
    "piq_timing_theorypeak", "piq_value_theorypeak", "y_timing_theorypeak", "y_value_theorypeak",
    "irate_timing_theorypeak", "irate_value_theorypeak", "rrate_timing_theorypeak", "rrate_value_theorypeak",
    "pi_timing_theorypeak", "pi_value_theorypeak",
    "piq_timing_agnpeak", "piq_value_agnpeak", "y_timing_agnpeak", "y_value_agnpeak",
    "irate_timing_agnpeak", "irate_value_agnpeak", "rrate_timing_agnpeak", "rrate_value_agnpeak",
    "pi_timing_agnpeak", "pi_value_agnpeak",
    "IScurve20", "infl_per_rr20", "Billsacrat20",
    "IScurve40", "infl_per_rr40", "Billsacrat40",
    "IScurve60", "infl_per_rr60", "Billsacrat60",
    "stky_pr_nondx", "stky_wg_nondx", "stky_all", "stky_pr_wg_ndx", "ndx_all", "ln_neq",
]
reg = reg[legacy_reg_columns].sort_values(["rule", "model"]).reset_index(drop=True)

legacy_reg_float32_columns = [
    "rule_tr", "rule_itr", "rule_g", "piq_value_min", "y_value_min", "irate_value_min",
    "rrate_value_min", "pi_value_min", "piq_value_max", "y_value_max", "irate_value_max",
    "rrate_value_max", "pi_value_max", "piq_timing_min", "y_timing_min", "irate_timing_min",
    "rrate_timing_min", "pi_timing_min", "piq_timing_max", "y_timing_max", "irate_timing_max",
    "rrate_timing_max", "pi_timing_max", "piq_cum20", "y_cum20", "irate_cum20",
    "rrate_cum20", "pi_cum20", "piq_cum40", "y_cum40", "irate_cum40", "rrate_cum40",
    "pi_cum40", "piq_cum60", "y_cum60", "irate_cum60", "rrate_cum60", "pi_cum60",
    "sacratio20", "sacratio40", "sacratio60", "irate_shock", "rrate_shock",
    "cb_authors_ext", "fiscal", "other_channel", "wg_ndx_mult", "wg_ndx_prprice",
    "wg_ndx_other", "stky_pr", "stky_pr_other", "stky_pr_rotemberg", "stky_pr_calvo",
    "not_stky_pr", "stky_pr_ndx", "stky_wg_ndx", "vint_early", "vint_mid", "vint_late",
    "est_early", "est_late", "piq_value_peak", "y_value_peak", "irate_value_peak",
    "rrate_value_peak", "pi_value_peak", "piq_timing_peak", "y_timing_peak",
    "irate_timing_peak", "rrate_timing_peak", "pi_timing_peak", "piq_timing_theorypeak",
    "piq_value_theorypeak", "y_timing_theorypeak", "y_value_theorypeak",
    "irate_timing_theorypeak", "irate_value_theorypeak", "rrate_timing_theorypeak",
    "rrate_value_theorypeak", "pi_timing_theorypeak", "pi_value_theorypeak",
    "piq_timing_agnpeak", "piq_value_agnpeak", "y_timing_agnpeak", "y_value_agnpeak",
    "irate_timing_agnpeak", "irate_value_agnpeak", "rrate_timing_agnpeak",
    "rrate_value_agnpeak", "pi_timing_agnpeak", "pi_value_agnpeak", "IScurve20",
    "infl_per_rr20", "Billsacrat20", "IScurve40", "infl_per_rr40", "Billsacrat40",
    "IScurve60", "infl_per_rr60", "Billsacrat60", "stky_pr_nondx", "stky_wg_nondx",
    "stky_all", "stky_pr_wg_ndx", "ndx_all", "ln_neq",
]
for col in legacy_reg_float32_columns:
    reg[col] = reg[col].astype(np.float32)
for horizon in horizons:
    reg[f"IScurve{horizon}"] = (reg[f"y_cum{horizon}"] / reg[f"rrate_cum{horizon}"]).astype(np.float32)
    reg[f"infl_per_rr{horizon}"] = (reg[f"piq_cum{horizon}"] / reg[f"rrate_cum{horizon}"]).astype(np.float32)
    reg[f"Billsacrat{horizon}"] = (reg[f"piq_cum{horizon}"] / reg[f"y_cum{horizon}"]).astype(np.float32)

reg.to_stata(output_dir / "MMB_reg_format.dta", write_index=False, version=118)
reg.to_excel(output_dir / "MMB_reg_format.xlsx", index=False)

In [55]:
print(f"Regression panel rows x columns: {reg.shape}")
display(reg.head())

Regression panel rows x columns: (222, 135)


,model,rule,rule_tr,rule_itr,rule_g,piq_value_min,y_value_min,irate_value_min,rrate_value_min,pi_value_min,...,Billsacrat40,IScurve60,infl_per_rr60,Billsacrat60,stky_pr_nondx,stky_wg_nondx,stky_all,stky_pr_wg_ndx,ndx_all,ln_neq
0,G2_SIGMA08,Growth,0.0,0.0,1.0,-7.700607e-03,-3.509796e-02,-0.969323,-1.145733,-7.439740e-03,...,1.190170,-0.293330,-0.355810,1.213005,0.0,0.0,1.0,1.0,1.0,4.276666
1,G3_CW03,Growth,0.0,0.0,1.0,-4.648895e-03,-4.516538e-02,-0.993665,-1.039293,-4.541145e-03,...,1.007909,-0.321913,-0.319579,0.992750,1.0,0.0,1.0,1.0,0.0,2.197225
2,G7_TAY93,Growth,0.0,0.0,1.0,3.730030e-04,1.336320e-04,-0.966616,-1.075820,3.967510e-04,...,0.648026,-0.874563,-0.582040,0.665520,1.0,1.0,1.0,0.0,0.0,2.639057
3,NK_AFL15,Growth,0.0,0.0,1.0,5.500000e-14,1.390000e-13,-0.856605,-1.110289,8.850000e-14,...,0.901697,-0.476297,-0.429474,0.901694,1.0,0.0,0.0,0.0,0.0,3.218876
4,NK_BGG99,Growth,0.0,0.0,1.0,-8.919438e-03,1.133134e-02,-0.825623,-1.653204,-8.907355e-03,...,0.259988,-2.413411,-0.509642,0.211171,1.0,0.0,0.0,0.0,0.0,2.833213


## Step 6: Build the cloud-graph IRF panel with model attributes and VAR benchmark.

In [56]:
irf_full = irf.loc[irf["rule"] != "Model"].copy()
irf_full = irf_full.drop(columns=["id"])
irf_full["period"] = irf_full["period"] - 1
irf_full = irf_full.loc[irf_full["period"] <= cloud_graph_extent].copy()
irf_full["id"] = irf_full.groupby(["model", "rule"], sort=True).ngroup() + 1
irf_full = irf_full.merge(rhs, on="model", how="left")
irf_full["_merge"] = 3
irf_full["model_type_n"] = np.nan
irf_full.loc[irf_full["calibrated"] == 1, "model_type_n"] = 1
irf_full.loc[irf_full["estimated"] == 1, "model_type_n"] = 2
irf_full.loc[(irf_full["calibrated"] == 1) & (irf_full["estimated"] == 1), "model_type_n"] = 3
irf_full["model_type"] = ""
irf_full.loc[irf_full["calibrated"] == 1, "model_type"] = "Calibrated"
irf_full.loc[irf_full["estimated"] == 1, "model_type"] = "Estimated"
irf_full.loc[(irf_full["calibrated"] == 1) & (irf_full["estimated"] == 1), "model_type"] = "Both"

bob = pd.read_csv(input_dir / "bob_var_irfs.csv")
bob.columns = [c.lower() for c in bob.columns]
bob = bob.rename(columns={"rtb": "irate", "pi": "piq"})
bob["model"] = "VAR, 1963:Q1-2007:Q4"
bob["rule"] = "NA"
bob["period"] = np.arange(len(bob))
for col in ["irate", "piq", "y"]:
    bob[col] = (-pd.to_numeric(bob[col], errors="coerce")).astype(np.float32).astype(float)
bob["rrate"] = (bob["irate"].astype(np.float32) - bob["piq"].shift(-1).astype(np.float32)).astype(np.float32).astype(float)
bob = bob.loc[bob["period"] <= cloud_graph_extent, ["period", "model", "rule", "piq", "y", "irate", "rrate"]].copy()
bob["pi"] = np.nan
bob["id"] = np.nan

for col in irf_full.columns:
    if col not in bob.columns:
        bob[col] = np.nan
bob = bob[irf_full.columns]
irf_full = pd.concat([irf_full, bob], ignore_index=True)
irf_full = irf_full.drop(columns=[c for c in ["cites", "cites_w1", "cites_w2", "cites_w3"] if c in irf_full.columns])
irf_full = irf_full.sort_values(["model", "rule", "period"]).reset_index(drop=True)
legacy_irf_full_columns = [
    "id", "period", "model", "rule", "piq", "y", "irate", "rrate", "pi",
    "cb_authors", "estimated", "calibrated", "neq", "open", "firm_bs", "bank",
    "hh_demand", "labor_frict", "gov_spend", "tax", "gov_debt", "learning", "stky_wg",
    "pr_ndx", "wg_ndx", "est_early", "est_late", "vint_early", "vint_mid",
    "vint_late", "pub_date", "est_start", "est_end", "stky_pr_calvo",
    "stky_pr_rotemberg", "stky_pr_other", "not_stky_pr", "stky_pr",
    "wg_ndx_mult", "wg_ndx_prprice", "wg_ndx_other", "stky_pr_ndx",
    "stky_pr_nondx", "stky_wg_ndx", "stky_wg_nondx", "stky_all",
    "stky_pr_wg_ndx", "ndx_all", "other_channel", "fiscal", "ln_neq",
    "cb_authors_ext", "_merge", "model_type_n", "model_type",
]
irf_full = irf_full[legacy_irf_full_columns]
legacy_irf_full_float32_columns = [
    "id", "period", "est_early", "est_late", "vint_early", "vint_mid",
    "vint_late", "stky_pr_calvo", "stky_pr_rotemberg", "stky_pr_other",
    "not_stky_pr", "stky_pr", "wg_ndx_mult", "wg_ndx_prprice", "wg_ndx_other",
    "stky_pr_ndx", "stky_pr_nondx", "stky_wg_ndx", "stky_wg_nondx",
    "stky_all", "stky_pr_wg_ndx", "ndx_all", "other_channel", "fiscal",
    "ln_neq", "cb_authors_ext", "model_type_n",
]
for col in legacy_irf_full_float32_columns:
    irf_full[col] = irf_full[col].astype(np.float32)
irf_full.to_stata(output_dir / "MMB_IRF_format_full.dta", write_index=False, version=118)

In [57]:
print(f"Cloud-graph IRF panel rows x columns: {irf_full.shape}")
display(irf_full.head())

Cloud-graph IRF panel rows x columns: (4683, 55)


,id,period,model,rule,piq,y,irate,rrate,pi,cb_authors,...,stky_all,stky_pr_wg_ndx,ndx_all,other_channel,fiscal,ln_neq,cb_authors_ext,_merge,model_type_n,model_type
0,1.0,0.0,G2_SIGMA08,Growth,0.107317,0.164931,-0.969323,-1.145733,0.026829,1.0,...,1.0,1.0,1.0,1.0,1.0,4.276666,1.0,3.0,1.0,Calibrated
1,1.0,1.0,G2_SIGMA08,Growth,0.176410,0.223661,-0.672435,-0.873348,0.070932,1.0,...,1.0,1.0,1.0,1.0,1.0,4.276666,1.0,3.0,1.0,Calibrated
2,1.0,2.0,G2_SIGMA08,Growth,0.200913,0.223788,-0.430918,-0.621122,0.121160,1.0,...,1.0,1.0,1.0,1.0,1.0,4.276666,1.0,3.0,1.0,Calibrated
3,1.0,3.0,G2_SIGMA08,Growth,0.190204,0.195972,-0.235425,-0.394057,0.168711,1.0,...,1.0,1.0,1.0,1.0,1.0,4.276666,1.0,3.0,1.0,Calibrated
4,1.0,4.0,G2_SIGMA08,Growth,0.158632,0.158585,-0.109285,-0.228717,0.181540,1.0,...,1.0,1.0,1.0,1.0,1.0,4.276666,1.0,3.0,1.0,Calibrated


## Step 7: Write text metadata for the stable binary outputs.

In [58]:
write_codebook(
    irf,
    output_dir / "MMB_IRF_format_codebook.txt",
    "MMB response CSVs imported through tasks/data/import_mmb_legacy_data",
    "Dropped duplicate, non-US calibrated, and excluded models listed in config/params.yaml; dropped model-rule responses; sign-flipped responses; period zero removed.",
)
write_codebook(
    irf_full,
    output_dir / "MMB_IRF_format_full_codebook.txt",
    "MMB_IRF_format.dta plus hand-coded model attributes and Bob VAR benchmark",
    f"Restricted to periods 0 through {cloud_graph_extent} after aligning the shock to period 0 for cloud graphs.",
)
write_codebook(
    reg,
    output_dir / "MMB_reg_format_codebook.txt",
    "Constructed IRF outcomes, sacrifice ratios, and hand-coded model attributes",
    "Merged horizons, sacrifice ratios, and RHS variables; model exclusions from config/params.yaml applied.",
)

print(f"Wrote IRF panel with {irf.shape[0]} rows and {irf.shape[1]} columns.")
print(f"Wrote cloud-graph IRF panel with {irf_full.shape[0]} rows and {irf_full.shape[1]} columns.")
print(f"Wrote regression panel with {reg.shape[0]} rows and {reg.shape[1]} columns.")

Wrote IRF panel with 21978 rows and 9 columns.
Wrote cloud-graph IRF panel with 4683 rows and 55 columns.
Wrote regression panel with 222 rows and 135 columns.
